## Objective

Build a campaign-level performance dataset for all campaigns that
started and ended within the 2025 calendar year.

### Grain
One row per campaign.

### Key business questions

- How many customers participated?
- What transaction activity was generated?
- How much reward cost was incurred?
- How many new customers were acquired?
- Which campaigns performed best relative to their targets and costs?

### Important analytical distinction

Observed campaign transaction value is not equivalent to incremental
business impact. Incrementality will be analyzed separately in a later
notebook.

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW campaign_scope AS

SELECT
    campaign_id,
    campaign_name,
    campaign_type,
    team,
    objective,
    start_date,
    end_date,

    -- Campaign duration
    DATEDIFF(end_date, start_date) + 1 AS campaign_days,

    budget,
    target_customer_count,
    target_transaction_count,
    target_transaction_amount

FROM `campaign&promotion`.gold.dim_campaign
WHERE start_date >= DATE '2025-01-01'
  AND end_date <= DATE '2025-12-31';

In [0]:
%sql

SELECT
    COUNT(*) AS campaign_count
FROM campaign_scope;

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW campaign_participation AS

SELECT
    campaign_id,

    COUNT(DISTINCT CASE
        WHEN eligible_flag = TRUE
        THEN customer_id
    END) AS eligible_customers,

    COUNT(DISTINCT CASE
        WHEN participated_flag = TRUE
        THEN customer_id
    END) AS participating_customers

FROM `campaign&promotion`.gold.fct_campaign_participation
WHERE campaign_id IN (
    SELECT campaign_id
    FROM campaign_scope
)
GROUP BY campaign_id;

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW campaign_participants_deduped AS
SELECT DISTINCT campaign_id, customer_id
FROM `campaign&promotion`.gold.fct_campaign_participation
WHERE participated_flag = TRUE;

CREATE OR REPLACE TEMP VIEW campaign_transactions AS
SELECT
    c.campaign_id,
    COUNT(DISTINCT t.transaction_id) AS transaction_count,
    COUNT(DISTINCT t.customer_id) AS transacting_customers,
    SUM(t.amount) AS transaction_value,
    SUM(t.fee) AS fee_revenue,
    SUM(t.cashback) AS transaction_cashback
FROM campaign_scope c
INNER JOIN campaign_participants_deduped p
    ON c.campaign_id = p.campaign_id
INNER JOIN `campaign&promotion`.gold.fct_transaction t
    ON p.customer_id = t.customer_id
   AND t.transaction_date BETWEEN c.start_date AND c.end_date
   AND t.status = 'SUCCESS'
GROUP BY c.campaign_id;

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW campaign_rewards AS

SELECT
    campaign_id,

    COUNT(DISTINCT redemption_id) AS redemption_count,

    COUNT(DISTINCT customer_id) AS rewarded_customers,

    SUM(reward_amount) AS reward_cost,

    SUM(cashback_amount) AS cashback_cost,

    SUM(discount_amount) AS discount_cost

FROM `campaign&promotion`.gold.fct_promotion_redemption

WHERE campaign_id IN (
    SELECT campaign_id
    FROM campaign_scope
)

GROUP BY campaign_id;

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW campaign_new_customers AS

SELECT
    c.campaign_id,

    COUNT(DISTINCT CASE
        WHEN d.registration_date BETWEEN c.start_date AND c.end_date
        THEN p.customer_id
    END) AS new_customers

FROM campaign_scope c

INNER JOIN `campaign&promotion`.gold.fct_campaign_participation p
    ON c.campaign_id = p.campaign_id
   AND p.participated_flag = TRUE

INNER JOIN `campaign&promotion`.gold.dim_customer d
    ON p.customer_id = d.customer_id

GROUP BY c.campaign_id;

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW campaign_performance AS

SELECT
    c.campaign_id,
    c.campaign_name,
    c.campaign_type,
    c.team,
    c.objective,
    c.start_date,
    c.end_date,
    c.campaign_days,

    -- Budget
    c.budget,

    -- Planned targets
    c.target_customer_count,
    c.target_transaction_count,
    c.target_transaction_amount,

    -- Participation
    COALESCE(p.eligible_customers, 0) AS eligible_customers,
    COALESCE(p.participating_customers, 0) AS participating_customers,

    ROUND(
        100.0 *
        COALESCE(p.participating_customers, 0)
        / NULLIF(p.eligible_customers, 0),
        2
    ) AS participation_rate_pct,

    -- Transactions
    COALESCE(t.transaction_count, 0) AS transaction_count,
    COALESCE(t.transacting_customers, 0) AS transacting_customers,
    COALESCE(t.transaction_value, 0) AS transaction_value,
    COALESCE(t.fee_revenue, 0) AS fee_revenue,

    ROUND(
        COALESCE(t.transaction_value, 0)
        / NULLIF(t.transaction_count, 0),
        2
    ) AS avg_transaction_value,

    -- Rewards
    COALESCE(r.redemption_count, 0) AS redemption_count,
    COALESCE(r.rewarded_customers, 0) AS rewarded_customers,
    COALESCE(r.reward_cost, 0) AS reward_cost,
    COALESCE(r.cashback_cost, 0) AS cashback_cost,
    COALESCE(r.discount_cost, 0) AS discount_cost,

    -- Acquisition
    COALESCE(n.new_customers, 0) AS new_customers,

    ROUND(
        COALESCE(r.reward_cost, 0)
        / NULLIF(n.new_customers, 0),
        2
    ) AS cac_reward_basis,

    -- Target achievement
    ROUND(
        100.0 *
        COALESCE(p.participating_customers, 0)
        / NULLIF(c.target_customer_count, 0),
        2
    ) AS customer_target_attainment_pct,

    ROUND(
        100.0 *
        COALESCE(t.transaction_count, 0)
        / NULLIF(c.target_transaction_count, 0),
        2
    ) AS transaction_target_attainment_pct,

    ROUND(
        100.0 *
        COALESCE(t.transaction_value, 0)
        / NULLIF(c.target_transaction_amount, 0),
        2
    ) AS transaction_value_target_attainment_pct,

    -- Observed efficiency, NOT true incremental ROI
    ROUND(
        COALESCE(t.transaction_value, 0)
        / NULLIF(r.reward_cost, 0),
        2
    ) AS transaction_value_to_reward_cost

FROM campaign_scope c

LEFT JOIN campaign_participation p
    ON c.campaign_id = p.campaign_id

LEFT JOIN campaign_transactions t
    ON c.campaign_id = t.campaign_id

LEFT JOIN campaign_rewards r
    ON c.campaign_id = r.campaign_id

LEFT JOIN campaign_new_customers n
    ON c.campaign_id = n.campaign_id;

In [0]:
%sql

SELECT *
FROM campaign_performance
ORDER BY transaction_value DESC;

### Campaign Ranking

In [0]:
%sql

-- Highest Transaction Value

SELECT
    campaign_id,
    campaign_name,
    campaign_type,
    transaction_value,
    reward_cost,
    participation_rate_pct
FROM campaign_performance
ORDER BY transaction_value DESC
LIMIT 10;

In [0]:
%sql
-- Highest Participation Rate
SELECT
    campaign_id,
    campaign_name,
    campaign_type,
    eligible_customers,
    participating_customers,
    participation_rate_pct
FROM campaign_performance
ORDER BY participation_rate_pct DESC
LIMIT 10;

In [0]:
%sql
-- Best Observed Efficiency
SELECT
    campaign_id,
    campaign_name,
    campaign_type,
    transaction_value,
    reward_cost,
    transaction_value_to_reward_cost
FROM campaign_performance
WHERE reward_cost > 0
ORDER BY transaction_value_to_reward_cost DESC
LIMIT 10;

In [0]:
%sql
-- Highest new-customer volume
SELECT
    campaign_id,
    campaign_name,
    campaign_type,
    new_customers,
    cac_reward_basis
FROM campaign_performance
ORDER BY new_customers DESC
LIMIT 10;

In [0]:
%sql

SELECT
    COUNT(*) AS campaign_count,
    COUNT(DISTINCT campaign_id) AS distinct_campaigns
FROM campaign_performance;

In [0]:
%sql

SELECT
    campaign_id,
    COUNT(*) AS rows_per_campaign
FROM campaign_performance
GROUP BY campaign_id
HAVING COUNT(*) > 1;

In [0]:
%sql
CREATE OR REPLACE TABLE `campaign&promotion`.gold.campaign_performance AS
SELECT * FROM campaign_performance;

In [0]:
%sql
select count(*) from `campaign&promotion`.gold.campaign_performance;